In [ ]:
from Models.fast_text.fast_text import FastText
from Datasets.datasets import MyDataset
from sklearn.model_selection import KFold
import copy

In [ ]:

import os
os.environ['NationalityDatasets'] = "./data/"

In [ ]:
dt = MyDataset()

#### Loading dataset

In [ ]:
data = dt.get_data()
train = dt.get_train()
test = dt.get_test()
validation = dt.get_validation()
train_validation = dt.get_train_validation()

In [ ]:
data.shape, train.shape, test.shape, validation.shape

In [ ]:
data.shape, train.shape, test.shape, validation.shape

In [ ]:
train.sample(n=10)

In [ ]:
data['level_1'].value_counts().plot.bar()

In [ ]:
train[train.country == 'others']

#### Classification

In [ ]:
region_1_ft = FastText(data=data, column='level_1')

In [ ]:
region_1_ft.save_train_test(train=train, test=test, validation=validation)

In [ ]:
region_1_ft.hyper_params = {
            "dim": 105,
            'lr': 0.1692979210449485,
            'seed': 0,
            'wordNgrams': 5,
            'maxn': 6,
            'minn': 3,
            'ws': 5,
}

In [ ]:
# %%timeit
# region_1_ft.training()

In [ ]:
# region_1_ft.save_model('level_1_classification.bin')

In [ ]:
region_1_ft.load_model('level_1_classification.bin')

In [ ]:
accuracy = region_1_ft.testing()
print('test accuracy: ', str(accuracy))

In [ ]:
region_1_ft.get_parameters()

In [ ]:
test_df = region_1_ft.load_df()
test_df['pred'] = region_1_ft.predict(test_df['text'])

In [ ]:
region_1_ft.print_classification_metrics(test_df['label'], test_df['pred'])

Micro F1-score 0.8363790592981472


Macro F1-score 0.8091636658756411


Weighted F1-score 0.8361904336367245


Weighted Precision 0.8375194323310818


Weighted Recall 0.8363790592981472


Accuracy Score 0.8363790592981472


Balanced Accuracy Score 0.793880658477273

In [ ]:
region_1_ft.print_classification_report(test_df['label'], test_df['pred'])

               precision    recall  f1-score   support

       German       0.76      0.84      0.80     74914
      English       0.80      0.89      0.84     58142
     Hispanic       0.89      0.88      0.89     58021
      Romance       0.79      0.82      0.80     57040
       Slavic       0.83      0.79      0.81     37330
Nordic-Baltic       0.89      0.78      0.83     29786
     EastAsia       0.91      0.82      0.86     23338
       Muslim       0.76      0.76      0.76     19167
    SouthAsia       0.85      0.73      0.79     12777
      African       0.79      0.54      0.64     12259
        Greek       0.81      0.61      0.70      3045
       Jewish       0.61      0.52      0.56      2046
       others       0.38      0.09      0.14      1262

     accuracy                           0.82    389127
    macro avg       0.77      0.70      0.72    389127
 weighted avg       0.82      0.82      0.81    389127

In [ ]:
cm = region_1_ft.confusion_matrix(test_df['label'], test_df['pred'], type='precision')
region_1_ft.display_confusion_matrix(cm)

#### Analysis

In [ ]:
test = test[['name', 'level_1', 'level_2', 'level_3',]]
test['label'] = test_df['label'].values
test['pred'] = test_df['pred'].values

In [ ]:
test[(test['label'] == 10) & (test['pred'] == 9)].level_3.value_counts()

In [ ]:
test[(test['level_3'] == 'Belgium')].pred.value_counts()

In [ ]:
region_1_ft.generate_label_encoding()
region_1_ft.label_encoding

5-fold cross validation

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)  
accuracies = []
accuracies2 = []
parameters = []
f1_micros = []
f1_macros = []
f1_weighteds = []
recalls = []
precisions = []
balanced_accuracies = []
cms = []
models = []
for train_index, val_index in kfold.split(train_validation):
    
    kfold_train = train_validation.iloc[train_index]
    kfold_validation = train_validation.iloc[val_index]
    print(kfold_train.shape, kfold_validation.shape)
    
    region_1_ft = FastText(data=data, column='level_1')
    region_1_ft.save_train_test(train=kfold_train, test=test, validation=kfold_validation)
    
    region_1_ft.training()
    accuracy = region_1_ft.testing()
    print('test accuracy: ', str(accuracy))
    accuracies.append(accuracy)
    parameters.append(region_1_ft.get_parameters())
    
    
    test_df = region_1_ft.load_df()
    test_df['pred'] = region_1_ft.predict(test_df['text'])
    
    region_1_ft.print_classification_metrics(test_df['label'], test_df['pred'])
    
    accuracies2.append(region_1_ft.accuracy_score(test_df['label'], test_df['pred']))
    f1_micros.append(region_1_ft.f1_micro_score(test_df['label'], test_df['pred']))
    f1_macros.append(region_1_ft.f1_macro_score(test_df['label'], test_df['pred']))
    f1_weighteds.append(region_1_ft.f1_weighted_score(test_df['label'], test_df['pred']))
    recalls.append(region_1_ft.recall_score(test_df['label'], test_df['pred']))
    precisions.append(region_1_ft.precision_score(test_df['label'], test_df['pred']))
    balanced_accuracies.append(region_1_ft.balanced_accuracy_score(test_df['label'], test_df['pred']))
    cms.append(region_1_ft.confusion_matrix(test_df['label'], test_df['pred'], type='precision'))
    
    region_1_ft.save_model(f'level_1_classification_{len(accuracies)}.bin')

In [ ]:
np.mean(accuracies)